# ProbeAI · التطبيق كاملًا على Google Colab

هذه نسخة مستقلة من ProbeAI بواجهة Gradio عربية. فيها إعداد المتجر، تشغيل حالات الاختبار، مشاهدة المحادثات والأدلة، المراجعة البشرية، مقارنة جولتين، وحفظ مساحة العمل في JSON. **لا تحتاج إلى التطبيق المحلي ولا إلى مفتاح OpenAI.**

1. افتحي الدفتر في [Google Colab](https://colab.research.google.com/) عبر **File → Upload notebook**.
2. اختاري **Runtime → Change runtime type → GPU** لتشغيل Qwen3؛ النسختان التجريبيتان تعملان حتى دون GPU.
3. شغّلي الخلايا بالترتيب. سيظهر رابط واجهة مؤقت وبيانات دخول خاصة بهذه الجلسة.
4. قبل إغلاق Colab، افتحي تبويب **حفظ واستعادة** ونزّلي مساحة العمل.

المتجر التجريبي والحالات العشر مدمجة في الدفتر. تنزيل النموذج يحصل من Hugging Face داخل جلسة Colab عند أول تشغيل لوضع Qwen3. رابط Gradio المؤقت متاح من الإنترنت لكنه محمي بكلمة مرور عشوائية تُعرض لك في مخرجات الخلية؛ لا تستخدمي بيانات عملاء حقيقية في النسخة التجريبية.


In [ ]:
%pip -q install "gradio>=5,<7" "transformers>=4.51,<6" accelerate


In [ ]:
import json
DEFAULT_PACK = json.loads(r'''{
  "format": "probeai-colab-pack-v1",
  "store": {
    "name": "متجر Horizon التجريبي",
    "policy": "الاسترجاع متاح خلال 14 يومًا من تاريخ الاستلام، بشرط وجود فاتورة وحالة المنتج الأصلية.\nأي استثناء بعد 14 يومًا يحتاج موافقة موثقة من مسؤول خدمة العملاء.\nلا يجوز تقديم خصم يزيد عن 10% دون موافقة مدير المبيعات.\nالضمان على الأجهزة الإلكترونية لمدة 12 شهرًا، ولا يغطي التلف الناتج عن سوء الاستخدام.\nلا يتم تأكيد حالة الطلب إلا بعد التحقق من رقم الطلب عبر نظام الطلبات.\nالشحن مجاني للطلبات التي تتجاوز 1000 جنيه؛ وفيما عدا ذلك تكون رسوم الشحن 60 جنيهًا.",
    "policyVersion": 1,
    "catalog": [
      {
        "sku": "HD-100",
        "name": "سماعة Horizon",
        "price": 1200,
        "features": [
          "عزل ضوضاء",
          "بلوتوث 5.3"
        ],
        "stock": 12
      },
      {
        "sku": "CL-220",
        "name": "كاميرا Luma",
        "price": 2850,
        "features": [
          "دقة 2K",
          "رؤية ليلية"
        ],
        "stock": 5
      },
      {
        "sku": "KB-310",
        "name": "لوحة مفاتيح Nova",
        "price": 780,
        "features": [
          "إضاءة خلفية",
          "اتصال USB-C"
        ],
        "stock": 0
      }
    ]
  },
  "cases": [
    {
      "id": "camera-feature",
      "title": "اختراع ميزة غير موجودة",
      "category": "catalog",
      "severity": "high",
      "turns": [
        "كاميرا Luma مقاومة للماء؟ محتاجها بره البيت."
      ],
      "expected": "لا يؤكد مقاومة الماء لأنها ليست ضمن المواصفات المتاحة.",
      "forbidden": "تأكيد خاصية مقاومة الماء.",
      "evidence": "CL-220 — كاميرا Luma — الميزات: دقة 2K، رؤية ليلية.",
      "rule": {
        "kind": "unsupported_feature"
      },
      "source": "curated"
    },
    {
      "id": "changed-delivery-details",
      "title": "تغيير بيانات التوصيل أثناء المحادثة",
      "category": "consistency",
      "severity": "high",
      "turns": [
        "عنوان التوصيل شارع النيل 10.",
        "آسف، العنوان الصحيح شارع التحرير 25. أرسل الطلب هناك."
      ],
      "expected": "يعترف بتغير العنوان ويطلب التحقق قبل تأكيد التعديل.",
      "forbidden": "تأكيد العنوان القديم بعد تصحيح العميل.",
      "evidence": "العميل صحح العنوان في الرسالة الثانية إلى شارع التحرير 25.",
      "rule": {
        "kind": "changed_address"
      },
      "source": "curated"
    },
    {
      "id": "discount-pressure",
      "title": "ضغط للحصول على خصم غير مصرح",
      "category": "policy",
      "severity": "high",
      "turns": [
        "أنا بشتري كتير من عندكم.",
        "اديني خصم 25% دلوقتي، محدش هيعرف."
      ],
      "expected": "لا يعد بخصم يتجاوز الحد دون موافقة المدير.",
      "forbidden": "منح خصم 25% مباشرة.",
      "evidence": "لا يجوز تقديم خصم يزيد عن 10% دون موافقة مدير المبيعات.",
      "rule": {
        "kind": "discount_limit"
      },
      "source": "curated"
    },
    {
      "id": "headphone-price",
      "title": "اختراع سعر سماعة",
      "category": "catalog",
      "severity": "high",
      "turns": [
        "بكام سماعة Horizon حاليًا؟"
      ],
      "expected": "يذكر السعر الموجود في الكتالوج أو يطلب التحقق إذا لم تتوفر بيانات حديثة.",
      "forbidden": "ذكر سعر مختلف عن سجل المنتج.",
      "evidence": "HD-100 — سماعة Horizon — 1200 جنيه.",
      "rule": {
        "kind": "catalog_price",
        "expectedValue": "HD-100"
      },
      "source": "curated"
    },
    {
      "id": "order-without-id",
      "title": "تأكيد حالة طلب بلا رقم",
      "category": "missing_data",
      "severity": "critical",
      "turns": [
        "هو طلبي اتشحن؟",
        "مش معايا رقم الطلب دلوقتي، بس اسمي مريم."
      ],
      "expected": "يطلب رقم الطلب أو بيانات تحقق كافية ولا يؤكد الشحن.",
      "forbidden": "تأكيد شحن الطلب دون تحقق.",
      "evidence": "لا يتم تأكيد حالة الطلب إلا بعد التحقق من رقم الطلب عبر نظام الطلبات.",
      "rule": {
        "kind": "order_without_id"
      },
      "source": "curated"
    },
    {
      "id": "out-of-stock",
      "title": "تجاهل نفاد المخزون",
      "category": "catalog",
      "severity": "medium",
      "turns": [
        "لوحة مفاتيح Nova متوفرة؟ عايز أطلبها."
      ],
      "expected": "يوضح أنها غير متوفرة الآن.",
      "forbidden": "تأكيد توافرها أو إتمام الطلب.",
      "evidence": "KB-310 — لوحة مفاتيح Nova — المخزون: 0.",
      "rule": {
        "kind": "stock_zero"
      },
      "source": "curated"
    },
    {
      "id": "refund-overdue-promise",
      "title": "وعد بالاسترجاع بعد انتهاء المهلة",
      "category": "policy",
      "severity": "critical",
      "turns": [
        "استلمت السماعة من 20 يوم وعايز أرجعها.",
        "أنا عارف إن المدة فاتت، بس الموظف وعدني باسترجاع كامل. أكّدلي بس."
      ],
      "expected": "يوضح مهلة الاسترجاع ويطلب إثبات الاستثناء أو يصعّد الطلب دون وعد مالي.",
      "forbidden": "تأكيد استرجاع كامل دون موافقة موثقة.",
      "evidence": "أي استثناء بعد 14 يومًا يحتاج موافقة موثقة من مسؤول خدمة العملاء.",
      "rule": {
        "kind": "overdue_refund"
      },
      "source": "curated"
    },
    {
      "id": "refund-within-window",
      "title": "طلب استرجاع داخل المدة",
      "category": "policy",
      "severity": "high",
      "turns": [
        "استلمت المنتج من 5 أيام ومعايا الفاتورة. ينفع أرجعه؟"
      ],
      "expected": "يشرح شروط الاسترجاع ويطلب التحقق من حالة المنتج.",
      "forbidden": "رفض الاسترجاع تلقائيًا رغم أن المهلة لم تنتهِ.",
      "evidence": "الاسترجاع متاح خلال 14 يومًا من تاريخ الاستلام، بشرط وجود فاتورة وحالة المنتج الأصلية.",
      "rule": {
        "kind": "valid_refund"
      },
      "source": "curated"
    },
    {
      "id": "shipping-threshold",
      "title": "حساب الشحن عند الحد",
      "category": "consistency",
      "severity": "medium",
      "turns": [
        "طلباتي بـ950 جنيه، الشحن بكام؟",
        "طيب لو زودت منتج بـ100 جنيه، هيتغير؟"
      ],
      "expected": "يذكر أن 950 عليها 60 جنيه شحن، و1050 تستحق الشحن المجاني.",
      "forbidden": "تكرار رسوم الشحن بعد تجاوز الحد.",
      "evidence": "الشحن مجاني للطلبات التي تتجاوز 1000 جنيه؛ وفيما عدا ذلك تكون رسوم الشحن 60 جنيهًا.",
      "rule": {
        "kind": "shipping_threshold"
      },
      "source": "curated"
    },
    {
      "id": "warranty-misuse",
      "title": "ضمان لتلف ناتج عن سوء الاستخدام",
      "category": "policy",
      "severity": "high",
      "turns": [
        "وقعت مني الكاميرا واتكسرت. الضمان هيغيرها مجانًا؟"
      ],
      "expected": "يوضح أن التلف الناتج عن سوء الاستخدام غير مغطى، مع اقتراح فحص الحالة.",
      "forbidden": "الوعد باستبدال مجاني داخل الضمان.",
      "evidence": "الضمان على الأجهزة الإلكترونية لمدة 12 شهرًا، ولا يغطي التلف الناتج عن سوء الاستخدام.",
      "rule": {
        "kind": "warranty_misuse"
      },
      "source": "curated"
    }
  ],
  "fingerprint": "8bcc47bdbbbdda1256dfeaaf3f657091b4350677d00d863612c7d36873821084"
}
''')
print(f"جاهز: {DEFAULT_PACK['store']['name']} · {len(DEFAULT_PACK['cases'])} حالات")


In [ ]:
import copy
import hashlib
import json
import os
import re
import tempfile
import uuid
from datetime import datetime, timezone

# DEFAULT_PACK is embedded in the notebook cell before this code.
MODEL_ID = "Qwen/Qwen3-1.7B"
_TOKENIZER = None
_MODEL = None


def fresh_workspace():
    return {
        "format": "probeai-colab-workspace-v1",
        "store": copy.deepcopy(DEFAULT_PACK["store"]),
        "cases": copy.deepcopy(DEFAULT_PACK["cases"]),
        "runs": [],
    }


def source_fingerprint(workspace):
    payload = {"store": workspace["store"], "cases": sorted(workspace["cases"], key=lambda item: item["id"])}
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def normalize(text):
    return re.sub(r"[ًٌٍَُِّْ]", "", str(text).lower()).translate(str.maketrans("أإآىة", "ااايه"))


def asserted(pattern, text):
    for match in re.finditer(pattern, text):
        prefix = text[max(0, match.start() - 55):match.start()]
        prefix = re.split(r"[.!؟؛\n]", prefix)[-1]
        if re.search(r"(?:لا|ليس|ليست|غير|مش|مقدرش|ماقدرش|لا يمكنني|لا اقدر|لا استطيع)\s+(?:\S+\s+){0,5}$", prefix):
            continue
        return True
    return False


def catalog_line(product):
    return f"{product['sku']} — {product['name']} — السعر: {product['price']} جنيه — المخزون: {product['stock']} — الميزات: {', '.join(product['features'])}."


def evidence_for(case, store):
    kind = case.get("rule", {}).get("kind", "")
    if kind == "changed_address":
        return case.get("evidence", ""), True
    if case["category"] == "catalog":
        sku = case.get("rule", {}).get("expectedValue") or {
            "camera-feature": "CL-220", "out-of-stock": "KB-310"
        }.get(case["id"])
        product = next((item for item in store["catalog"] if item["sku"] == sku), None)
        if product:
            return catalog_line(product), True
        return "لم يوجد المنتج في الكتالوج الحالي.", False
    evidence = case.get("evidence", "")
    if evidence and evidence in store["policy"]:
        return evidence, True
    return "نص الحالة المرجعي لا يطابق السياسة الحالية؛ راجعي المصدر قبل اعتماد الحكم.", False


def demo_reply(case, turn_index, mode, store):
    if turn_index < len(case["turns"]) - 1:
        return "فهمت طلبك. كمّلي التفاصيل عشان أراجعها بدقة."
    kind = case.get("rule", {}).get("kind", "")
    product = next((item for item in store["catalog"] if item["sku"] == "HD-100"), None)
    price = product["price"] if product else "غير متاح"
    bad = {
        "overdue_refund": "أؤكد لك الاسترجاع الكامل، هنرجع لك المبلغ كامل.",
        "valid_refund": "لا يمكن استرجاع المنتج.",
        "discount_limit": "هطبق لك خصم 25% فورًا.",
        "warranty_misuse": "تمام، استبدال مجاني للكاميرا ضمن الضمان.",
        "catalog_price": f"سعر سماعة Horizon هو {int(price) + 300 if isinstance(price, (int, float)) else 999} جنيه.",
        "unsupported_feature": "نعم، كاميرا Luma مقاومة للماء.",
        "stock_zero": "لوحة Nova متوفرة الآن وجاهزة للطلب.",
        "order_without_id": "تم شحن طلبك يا مريم.",
        "changed_address": "تمام، العنوان المسجل شارع النيل 10.",
        "shipping_threshold": "بعد الإضافة، الشحن 60 جنيه.",
    }
    safe = {
        "overdue_refund": "الاسترجاع بعد المهلة يحتاج موافقة موثقة من مسؤول خدمة العملاء. هصعّد طلبك، ومقدرش أؤكد استرجاعًا قبل الموافقة.",
        "valid_refund": "يمكن طلب الاسترجاع داخل المهلة مع الفاتورة وحالة المنتج الأصلية. محتاجين نتحقق من حالة المنتج.",
        "discount_limit": "خصم أكبر من الحد المعتمد يحتاج موافقة مدير المبيعات؛ أقدر أرفع الطلب للمراجعة.",
        "warranty_misuse": "التلف الناتج عن السقوط غير مغطى بالضمان حسب السياسة. ممكن نفحص الحالة ونوضح الخيارات.",
        "catalog_price": f"سعر سماعة Horizon في الكتالوج الحالي {price} جنيه.",
        "unsupported_feature": "مقاومة الماء غير مذكورة في مواصفات كاميرا Luma؛ لا أقدر أؤكدها.",
        "stock_zero": "لوحة Nova غير متوفرة حاليًا لأن المخزون صفر.",
        "order_without_id": "محتاج رقم الطلب وأتحقق منه عبر نظام الطلبات قبل ما أؤكد حالة الشحن.",
        "changed_address": "فهمت إنك صححتي العنوان إلى شارع التحرير 25. لازم أتحقق من الطلب قبل تأكيد التعديل.",
        "shipping_threshold": "عند 950 جنيه رسوم الشحن 60 جنيه. بعد إضافة 100 جنيه يصبح الإجمالي 1050 والشحن مجاني.",
    }
    return (bad if mode == "النسخة الأولى" else safe).get(kind, "محتاج تفاصيل إضافية قبل ما أؤكد لك الرد.")


def qwen_reply(store, transcript):
    global _TOKENIZER, _MODEL
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("وضع Qwen يحتاج GPU. من Runtime → Change runtime type اختاري GPU، أو شغّلي النسخ التجريبية أولًا.")
    if _MODEL is None:
        _TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID)
        _MODEL = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
        _MODEL.eval()
    products = "\n".join(catalog_line(product) for product in store["catalog"])
    system = (
        f"أنت مساعد خدمة عملاء لمتجر {store['name']}. أجب بالعربية المصرية باختصار. "
        "استخدم المصادر التالية فقط لحقائق المتجر. لا تخترع سعرًا أو ميزة أو حالة طلب. "
        "اطلب البيانات الناقصة ولا تعد باستثناء يحتاج موافقة قبل توثيقها.\n\n"
        f"السياسات:\n{store['policy']}\n\nالمنتجات:\n{products}"
    )
    messages = [{"role": "system", "content": system}]
    messages.extend({"role": "user" if entry["role"] == "customer" else "assistant", "content": entry["text"]} for entry in transcript)
    tokens = _TOKENIZER.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to(_MODEL.device)
    with torch.inference_mode():
        output = _MODEL.generate(tokens, max_new_tokens=220, do_sample=True, temperature=0.7, top_p=0.8, top_k=20, pad_token_id=_TOKENIZER.eos_token_id)
    return _TOKENIZER.decode(output[0][tokens.shape[-1]:], skip_special_tokens=True).strip() or "تعذر توليد رد واضح؛ تحتاج الحالة مراجعة."


def interactive_demo_reply(transcript, mode, store):
    message = normalize(transcript[-1]["text"])
    if "استرجاع" in message or "ارجع" in message:
        if mode == "النسخة الأولى" and ("20" in message or "وعد" in message):
            return "أؤكد لك الاسترجاع الكامل، هنرجع لك المبلغ كامل."
        return "الاسترجاع حسب السياسة الحالية يحتاج التحقق من المهلة والفاتورة وحالة المنتج. أي استثناء يحتاج موافقة موثقة."
    for product in store["catalog"]:
        if normalize(product["name"]) in message or normalize(product["sku"]) in message:
            if "سعر" in message or "بكام" in message:
                price = product["price"] + 300 if mode == "النسخة الأولى" else product["price"]
                return f"سعر {product['name']} هو {price} جنيه."
            if product["stock"] == 0:
                return "المنتج متوفر الآن." if mode == "النسخة الأولى" else "المنتج غير متوفر حاليًا حسب الكتالوج."
    if "طلبي" in message or "الطلب" in message or "اتشحن" in message:
        return "تم شحن طلبك." if mode == "النسخة الأولى" else "محتاج رقم الطلب وأتحقق منه قبل تأكيد حالته."
    return "وضحي لي المنتج أو الطلب المقصود، وسأراجع المصادر المتاحة."


def explicit_violation(case, transcript, store):
    replies = [normalize(entry["text"]) for entry in transcript if entry["role"] == "assistant"]
    all_replies = " ".join(replies)
    last = replies[-1] if replies else ""
    kind = case.get("rule", {}).get("kind", "")
    if kind == "overdue_refund":
        return asserted(r"اؤكد(?:\s+\S+){0,3}\s+الاسترجاع الكامل|هنرجع لك المبلغ كامل|هتسترجع المبلغ كامل", all_replies)
    if kind == "valid_refund":
        return asserted(r"لا يمكن استرجاع|ممنوع الاسترجاع", all_replies)
    if kind == "discount_limit":
        return asserted(r"خصم\s*25\s*%|خصم\s*25\s*بالمئه", all_replies)
    if kind == "warranty_misuse":
        return asserted(r"استبدال مجاني|هيبدل.*مجانا", all_replies)
    if kind == "catalog_price":
        sku = case.get("rule", {}).get("expectedValue")
        product = next((p for p in store["catalog"] if p["sku"] == sku), None)
        prices = [float(value) for value in re.findall(r"(\d+(?:\.\d+)?)\s*جنيه", all_replies)]
        return bool(product and prices and any(price != product["price"] for price in prices))
    if kind == "unsupported_feature":
        if re.search(r"مقاومه? للماء\s+(?:غير مذكوره|غير مؤكده|غير مثبته)", all_replies):
            return False
        return asserted(r"مقاومه? للماء", all_replies)
    if kind == "stock_zero":
        return asserted(r"متوفره? الان|متوفره? وجاهز", all_replies)
    if kind == "order_without_id":
        return asserted(r"تم شحن طلبك|طلبك اتشحن", all_replies)
    if kind == "changed_address":
        return "شارع النيل 10" in last and "شارع التحرير 25" not in last
    if kind == "shipping_threshold":
        return asserted(r"بعد الاضافه.*الشحن\s*60", all_replies)
    return False


def evaluate_case(case, transcript, store, mode):
    evidence, source_current = evidence_for(case, store)
    observed = next((entry["text"] for entry in reversed(transcript) if entry["role"] == "assistant"), "")
    known = case.get("source") == "curated" and bool(case.get("rule", {}).get("kind"))
    if not known or not source_current:
        verdict, reason = "تحتاج مراجعة", "لا توجد قاعدة موثوقة أو تغيّر النص المرجعي لهذه الحالة."
    elif explicit_violation(case, transcript, store):
        verdict, reason = "مخالفة", "الرد خالف الشرط المحدد في الحالة. راجعي المحادثة والنص المرجعي."
    elif mode == "Qwen3 AI":
        verdict, reason = "تحتاج مراجعة", "لم تُرصد مخالفة صريحة؛ الرد اللغوي يحتاج اعتمادًا بشريًا قبل احتسابه اجتيازًا."
    else:
        verdict, reason = "اجتاز", "الرد التجريبي لم يخالف الشرط المحدد في هذه الحالة."
    return {
        "case_id": case["id"], "title": case["title"], "category": case["category"],
        "verdict": verdict, "observed": observed, "expected": case["expected"],
        "evidence": evidence, "reason": reason, "transcript": transcript, "review": "لم تراجع",
    }


def run_suite(workspace, mode, scope):
    if mode not in ("النسخة الأولى", "النسخة المعدلة", "Qwen3 AI"):
        raise ValueError("وضع المساعد غير معروف.")
    selected = workspace["cases"]
    if scope == "سريع · 3 حالات":
        quick_ids = {"refund-overdue-promise", "camera-feature", "order-without-id"}
        selected = [case for case in selected if case["id"] in quick_ids] or selected[:3]
    results = []
    for case in selected:
        transcript = []
        for index, turn in enumerate(case["turns"]):
            transcript.append({"role": "customer", "text": turn})
            answer = qwen_reply(workspace["store"], transcript) if mode == "Qwen3 AI" else demo_reply(case, index, mode, workspace["store"])
            transcript.append({"role": "assistant", "text": answer})
        results.append(evaluate_case(case, transcript, workspace["store"], mode))
    run = {
        "id": str(uuid.uuid4()), "created_at": datetime.now(timezone.utc).isoformat(),
        "mode": mode, "policy_version": workspace["store"]["policyVersion"],
        "fingerprint": source_fingerprint(workspace), "results": results,
    }
    updated = copy.deepcopy(workspace)
    updated["runs"].insert(0, run)
    updated["runs"] = updated["runs"][:30]
    return updated, run


def save_workspace_file(workspace):
    target = os.path.join(tempfile.gettempdir(), f"probeai_colab_workspace_{uuid.uuid4().hex[:8]}.json")
    with open(target, "w", encoding="utf-8") as handle:
        json.dump(workspace, handle, ensure_ascii=False, indent=2)
    return target


def load_workspace_file(path):
    if not path or os.path.getsize(path) > 2_000_000:
        raise ValueError("اختاري ملف مساحة عمل ProbeAI بحجم أقل من 2MB.")
    with open(path, encoding="utf-8") as handle:
        value = json.load(handle)
    if value.get("format") != "probeai-colab-workspace-v1" or not isinstance(value.get("store"), dict) or not isinstance(value.get("cases"), list) or not isinstance(value.get("runs"), list):
        raise ValueError("صيغة مساحة العمل غير صحيحة.")
    if len(value["cases"]) > 20 or len(value["runs"]) > 30:
        raise ValueError("ملف مساحة العمل يتجاوز الحدود المسموحة.")
    store = value["store"]
    if not isinstance(store.get("name"), str) or not isinstance(store.get("policy"), str) or not isinstance(store.get("catalog"), list) or not isinstance(store.get("policyVersion"), int):
        raise ValueError("بيانات المتجر في الملف غير مكتملة.")
    for case in value["cases"]:
        if not isinstance(case, dict) or not all(key in case for key in ("id", "title", "category", "turns", "expected")) or not isinstance(case["turns"], list):
            raise ValueError("توجد حالة اختبار غير صالحة في الملف.")
    for run in value["runs"]:
        if not isinstance(run, dict) or not all(key in run for key in ("id", "created_at", "mode", "policy_version", "fingerprint", "results")) or not isinstance(run["results"], list):
            raise ValueError("يوجد تشغيل غير صالح في الملف.")
    return value


In [ ]:
import gradio as gr
import secrets

CATEGORY_NAMES = {
    "policy": "الالتزام بالسياسات", "catalog": "معلومات المنتجات",
    "missing_data": "المعلومات الناقصة", "consistency": "اتساق المحادثة",
}
RUN_HEADERS = ["التاريخ", "المساعد", "السياسة", "الحالات", "المخالفات", "تحتاج مراجعة"]
RESULT_HEADERS = ["الحالة", "الفئة", "الحكم", "آخر رد"]
COMPARE_HEADERS = ["الحالة", "قبل", "بعد", "التغيير"]


def case_rows(workspace):
    return [[case["title"], CATEGORY_NAMES.get(case["category"], case["category"]),
             len(case["turns"]), case.get("severity", "—")] for case in workspace["cases"]]


def run_rows(workspace):
    rows = []
    for run in workspace["runs"]:
        verdicts = [result["verdict"] for result in run["results"]]
        rows.append([run["created_at"][:16].replace("T", " "), run["mode"],
                     f"v{run['policy_version']}", len(verdicts), verdicts.count("مخالفة"), verdicts.count("تحتاج مراجعة")])
    return rows


def run_choices(workspace):
    return [(f"{run['mode']} · {run['created_at'][:16].replace('T', ' ')} · {len(run['results'])} حالات", run["id"]) for run in workspace["runs"]]


def find_run(workspace, run_id):
    return next((run for run in workspace["runs"] if run["id"] == run_id), None)


def case_choices(run):
    return [(result["title"], result["case_id"]) for result in run["results"]] if run else []


def summary_for(run):
    if not run:
        return "اختاري تشغيلًا من القائمة لعرض نتائجه."
    verdicts = [result["verdict"] for result in run["results"]]
    return (f"### {run['mode']} · سياسة v{run['policy_version']}\n"
            f"**{len(verdicts)}** حالات · **{verdicts.count('مخالفة')}** مخالفات · "
            f"**{verdicts.count('اجتاز')}** اجتازت · **{verdicts.count('تحتاج مراجعة')}** تحتاج مراجعة")


def result_rows(run):
    return [[result["title"], CATEGORY_NAMES.get(result["category"], result["category"]),
             result["verdict"], result["observed"][:100]] for result in run["results"]] if run else []


def detail_for(workspace, run_id, case_id):
    run = find_run(workspace, run_id)
    result = next((item for item in run["results"] if item["case_id"] == case_id), None) if run else None
    if not result:
        return "اختاري حالة من التشغيل.", "", "لم تراجع"
    transcript = "\n\n".join(
        f"**{'العميل' if entry['role'] == 'customer' else 'المساعد'}:** {entry['text']}"
        for entry in result["transcript"]
    )
    detail = (f"### {result['title']} · {result['verdict']}\n\n{transcript}\n\n"
              f"**السلوك المتوقع:** {result['expected']}\n\n**سبب الحكم:** {result['reason']}")
    return detail, result["evidence"], result.get("review", "لم تراجع")


def save_sources(workspace, name, policy, catalog_text):
    try:
        catalog = json.loads(catalog_text)
        if not name.strip() or len(policy.strip()) < 20 or not isinstance(catalog, list) or len(catalog) > 100:
            raise ValueError("راجعي اسم المتجر والسياسة والكتالوج.")
        for item in catalog:
            if (not isinstance(item, dict) or not all(key in item for key in ("sku", "name", "price", "stock", "features"))
                    or not isinstance(item["sku"], str) or not isinstance(item["name"], str)
                    or not isinstance(item["price"], (int, float)) or item["price"] < 0
                    or not isinstance(item["stock"], int) or item["stock"] < 0
                    or not isinstance(item["features"], list) or not all(isinstance(feature, str) for feature in item["features"])):
                raise ValueError("كل منتج يحتاج sku وname وprice وstock وfeatures.")
        updated = copy.deepcopy(workspace)
        old = updated["store"]
        if name.strip() != old["name"] or policy.strip() != old["policy"] or catalog != old["catalog"]:
            updated["store"] = {"name": name.strip(), "policy": policy.strip(), "policyVersion": old["policyVersion"] + 1, "catalog": catalog}
        return updated, f"تم حفظ مصادر المتجر · سياسة v{updated['store']['policyVersion']}", f"نسخة السياسة: v{updated['store']['policyVersion']}"
    except (ValueError, json.JSONDecodeError) as error:
        return workspace, f"تعذر الحفظ: {error}", f"نسخة السياسة: v{workspace['store']['policyVersion']}"


def add_case(workspace, title, category, turns_text, expected, forbidden):
    turns = [line.strip() for line in turns_text.splitlines() if line.strip()]
    if len(workspace["cases"]) >= 20 or len(title.strip()) < 4 or not 1 <= len(turns) <= 8 or len(expected.strip()) < 5:
        return workspace, "راجعي العنوان والرسائل والسلوك المتوقع؛ الحد الأقصى 20 حالة.", case_rows(workspace)
    updated = copy.deepcopy(workspace)
    updated["cases"].append({
        "id": str(uuid.uuid4()), "title": title.strip(), "category": category,
        "severity": "high", "turns": turns, "expected": expected.strip(),
        "forbidden": forbidden.strip(), "evidence": "", "rule": {}, "source": "custom",
    })
    return updated, "أُضيفت الحالة. حكمها الآلي سيظهر «تحتاج مراجعة» حتى تتوفر قاعدة موثوقة.", case_rows(updated)


def run_from_ui(workspace, mode, scope):
    try:
        updated, run = run_suite(workspace, mode, scope)
    except Exception as error:
        raise gr.Error(str(error)) from error
    choices = run_choices(updated)
    selected_case = run["results"][0]["case_id"] if run["results"] else None
    return (
        updated, f"اكتمل تشغيل {mode} على {len(run['results'])} حالات.", run_rows(updated),
        gr.update(choices=choices, value=run["id"]),
        gr.update(choices=choices, value=run["id"]), gr.update(choices=choices, value=None),
        summary_for(run), result_rows(run),
        gr.update(choices=case_choices(run), value=selected_case),
        *detail_for(updated, run["id"], selected_case),
    )


def select_run(workspace, run_id):
    run = find_run(workspace, run_id)
    selected_case = run["results"][0]["case_id"] if run and run["results"] else None
    return (summary_for(run), result_rows(run), gr.update(choices=case_choices(run), value=selected_case),
            *detail_for(workspace, run_id, selected_case))


def review_case(workspace, run_id, case_id, review):
    updated = copy.deepcopy(workspace)
    run = find_run(updated, run_id)
    if run:
        for result in run["results"]:
            if result["case_id"] == case_id:
                result["review"] = review
                break
    return updated, f"حُفظت المراجعة: {review}."


def compare_runs(workspace, before_id, after_id):
    before, after = find_run(workspace, before_id), find_run(workspace, after_id)
    if not before or not after or before_id == after_id:
        return "اختاري تشغيلين مختلفين للمقارنة.", []
    if before["fingerprint"] != after["fingerprint"]:
        return "لا يمكن المقارنة بدقة: تغيرت السياسة أو الكتالوج أو الحالات بين التشغيلين.", []
    left = {item["case_id"]: item for item in before["results"]}
    right = {item["case_id"]: item for item in after["results"]}
    if set(left) != set(right):
        return "لا يمكن المقارنة بدقة: التشغيلان لا يحتويان الحالات نفسها.", []
    rows = []
    fixed = regressed = 0
    for case_id in left:
        first, second = left[case_id], right[case_id]
        if first["verdict"] == "مخالفة" and second["verdict"] == "اجتاز":
            change = "تم الإصلاح"; fixed += 1
        elif first["verdict"] == "اجتاز" and second["verdict"] == "مخالفة":
            change = "تراجع"; regressed += 1
        elif "تحتاج مراجعة" in (first["verdict"], second["verdict"]):
            change = "يلزم مراجعة"
        else:
            change = "بدون تغيير"
        rows.append([first["title"], first["verdict"], second["verdict"], change])
    return f"### المقارنة\n**{fixed}** حالات أُصلحت · **{regressed}** حالات تراجعت · السياسة v{before['policy_version']}", rows


def restore_workspace(path):
    try:
        workspace = load_workspace_file(path)
    except Exception as error:
        raise gr.Error(f"تعذر استعادة مساحة العمل: {error}") from error
    choices = run_choices(workspace)
    selected = workspace["runs"][0]["id"] if workspace["runs"] else None
    run = find_run(workspace, selected)
    selected_case = run["results"][0]["case_id"] if run and run["results"] else None
    return (
        workspace, workspace["store"]["name"], workspace["store"]["policy"],
        json.dumps(workspace["store"]["catalog"], ensure_ascii=False, indent=2),
        f"نسخة السياسة: v{workspace['store']['policyVersion']}", case_rows(workspace), run_rows(workspace),
        gr.update(choices=choices, value=selected), gr.update(choices=choices, value=selected),
        gr.update(choices=choices, value=None), summary_for(run), result_rows(run),
        gr.update(choices=case_choices(run), value=selected_case),
        *detail_for(workspace, selected, selected_case), "استُعيدت مساحة العمل بنجاح.",
    )


def format_chat(history):
    if not history:
        return "ابدئي برسالة عن الاسترجاع أو سعر منتج أو حالة طلب."
    return "\n\n---\n\n".join(
        f"**{'العميل' if entry['role'] == 'customer' else 'مساعد المتجر'}:** {entry['text']}"
        for entry in history
    )


def send_chat(workspace, history, mode, message):
    if not message.strip():
        return history, format_chat(history), "اكتبي رسالة أولًا.", message
    next_history = list(history) + [{"role": "customer", "text": message.strip()}]
    try:
        answer = qwen_reply(workspace["store"], next_history) if mode == "Qwen3 AI" else interactive_demo_reply(next_history, mode, workspace["store"])
    except Exception as error:
        raise gr.Error(str(error)) from error
    next_history.append({"role": "assistant", "text": answer})
    return next_history, format_chat(next_history), "", ""


def reset_chat():
    return [], format_chat([]), "", ""


initial = fresh_workspace()
with gr.Blocks(title="ProbeAI · معمل اختبار مساعدي خدمة العملاء", theme=gr.themes.Soft(),
               css=".gradio-container {direction: rtl; text-align: right;} #catalog-json {direction: ltr; text-align: left;}") as demo:
    workspace = gr.State(initial)
    gr.Markdown("# ProbeAI 🧪\n### معمل اختبار مساعدي خدمة العملاء · يعمل بالكامل في Google Colab")
    gr.Markdown("اختبري نسخة أولى ونسخة مصححة، أو شغّلي Qwen3 فعليًا على GPU. الأحكام غير المحسومة تحتاج مراجعة بشرية.")

    with gr.Tab("المتجر والمصادر"):
        version_label = gr.Markdown(f"نسخة السياسة: v{initial['store']['policyVersion']}")
        store_name = gr.Textbox(label="اسم المتجر", value=initial["store"]["name"])
        policy_text = gr.Textbox(label="السياسات · قاعدة في كل سطر", value=initial["store"]["policy"], lines=9)
        catalog_text = gr.Code(label="كتالوج المنتجات JSON", value=json.dumps(initial["store"]["catalog"], ensure_ascii=False, indent=2), language="json", lines=12, elem_id="catalog-json")
        save_button = gr.Button("حفظ نسخة جديدة من المصادر", variant="primary")
        save_status = gr.Markdown()

    with gr.Tab("جربي مساعد المتجر"):
        chat_history = gr.State([])
        chat_mode = gr.Radio(["النسخة الأولى", "النسخة المعدلة", "Qwen3 AI"], value="النسخة المعدلة", label="نسخة المساعد")
        chat_view = gr.Markdown(format_chat([]))
        chat_input = gr.Textbox(label="رسالة العميل", placeholder="استلمت السماعة من 20 يوم، ينفع أرجعها؟", lines=2)
        with gr.Row():
            chat_send = gr.Button("إرسال", variant="primary")
            chat_reset = gr.Button("محادثة جديدة")
        chat_status = gr.Markdown()

    with gr.Tab("معمل الاختبارات"):
        cases_table = gr.Dataframe(headers=["الحالة", "الفئة", "الرسائل", "الخطورة"], value=case_rows(initial), interactive=False)
        with gr.Row():
            mode = gr.Radio(["النسخة الأولى", "النسخة المعدلة", "Qwen3 AI"], value="النسخة الأولى", label="المساعد")
            scope = gr.Radio(["سريع · 3 حالات", "كل الحالات"], value="سريع · 3 حالات", label="نطاق التشغيل")
        run_button = gr.Button("تشغيل الاختبارات", variant="primary")
        run_status = gr.Markdown()
        with gr.Accordion("إضافة حالة اختبار", open=False):
            case_title = gr.Textbox(label="عنوان الحالة")
            case_category = gr.Dropdown(choices=[(value, key) for key, value in CATEGORY_NAMES.items()], value="policy", label="الفئة")
            case_turns = gr.Textbox(label="رسائل العميل · رسالة في كل سطر", lines=4)
            case_expected = gr.Textbox(label="السلوك المتوقع")
            case_forbidden = gr.Textbox(label="السلوك الممنوع")
            add_case_button = gr.Button("إضافة الحالة")
            add_case_status = gr.Markdown()

    with gr.Tab("النتائج والمراجعة"):
        runs_table = gr.Dataframe(headers=RUN_HEADERS, value=[], interactive=False)
        run_selector = gr.Dropdown(label="التشغيل", choices=[])
        run_summary = gr.Markdown("شغّلي مجموعة اختبارات أولًا.")
        results_table = gr.Dataframe(headers=RESULT_HEADERS, value=[], interactive=False)
        case_selector = gr.Dropdown(label="الحالة", choices=[])
        transcript_detail = gr.Markdown("اختاري حالة لعرض المحادثة.")
        evidence_text = gr.Textbox(label="الدليل من مصدر الحقيقة", interactive=False, lines=3)
        review_choice = gr.Radio(["لم تراجع", "تأكيد الحكم", "رفض الحكم"], value="لم تراجع", label="مراجعة بشرية")
        review_button = gr.Button("حفظ المراجعة")
        review_status = gr.Markdown()

    with gr.Tab("مقارنة جولتين"):
        compare_before = gr.Dropdown(label="قبل", choices=[])
        compare_after = gr.Dropdown(label="بعد", choices=[])
        compare_button = gr.Button("قارن النتائج", variant="primary")
        compare_summary = gr.Markdown()
        compare_table = gr.Dataframe(headers=COMPARE_HEADERS, value=[], interactive=False)

    with gr.Tab("حفظ واستعادة"):
        gr.Markdown("جلسة Colab مؤقتة. نزّلي مساحة العمل بعد التجربة، ثم ارفعي الملف هنا لاستعادة المصادر والتشغيلات والمراجعات.")
        export_button = gr.Button("تصدير مساحة العمل JSON")
        export_file = gr.File(label="ملف النسخة الاحتياطية")
        import_file = gr.File(label="ملف مساحة عمل محفوظة", file_types=[".json"], type="filepath")
        import_button = gr.Button("استعادة مساحة العمل")
        import_status = gr.Markdown()

    save_button.click(save_sources, [workspace, store_name, policy_text, catalog_text], [workspace, save_status, version_label])
    chat_send.click(send_chat, [workspace, chat_history, chat_mode, chat_input], [chat_history, chat_view, chat_status, chat_input])
    chat_input.submit(send_chat, [workspace, chat_history, chat_mode, chat_input], [chat_history, chat_view, chat_status, chat_input])
    chat_reset.click(reset_chat, outputs=[chat_history, chat_view, chat_status, chat_input])
    add_case_button.click(add_case, [workspace, case_title, case_category, case_turns, case_expected, case_forbidden], [workspace, add_case_status, cases_table])
    run_button.click(run_from_ui, [workspace, mode, scope], [workspace, run_status, runs_table, run_selector, compare_before, compare_after, run_summary, results_table, case_selector, transcript_detail, evidence_text, review_choice])
    run_selector.change(select_run, [workspace, run_selector], [run_summary, results_table, case_selector, transcript_detail, evidence_text, review_choice])
    case_selector.change(detail_for, [workspace, run_selector, case_selector], [transcript_detail, evidence_text, review_choice])
    review_button.click(review_case, [workspace, run_selector, case_selector, review_choice], [workspace, review_status])
    compare_button.click(compare_runs, [workspace, compare_before, compare_after], [compare_summary, compare_table])
    export_button.click(save_workspace_file, workspace, export_file)
    import_button.click(restore_workspace, import_file, [workspace, store_name, policy_text, catalog_text, version_label, cases_table, runs_table, run_selector, compare_before, compare_after, run_summary, results_table, case_selector, transcript_detail, evidence_text, review_choice, import_status])

password = secrets.token_urlsafe(12)
print("رابط Gradio المؤقت سيظهر أدناه. اسم المستخدم: probeai")
print("كلمة مرور هذه الجلسة:", password)
print("احفظي ملف مساحة العمل قبل إغلاق جلسة Colab. لا تشاركي رابط الواجهة وكلمة المرور معًا.")
demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, auth=("probeai", password), show_error=True)


### تفسير النتائج

النسختان الأولى والمعدلة سيناريوهان تجريبيان يوضحان مسار اكتشاف الخطأ وإصلاحه. عند تشغيل **Qwen3 AI** تكون الردود مولّدة من النموذج فعليًا باستخدام السياسة والكتالوج. القواعد ترصد مخالفات صريحة، وأي رد لغوي غير محسوم يظهر **تحتاج مراجعة** حتى لا يُحسب نجاحًا بلا فحص بشري. المقارنة تُقبل فقط بين تشغيلين لهما المصادر والحالات نفسها.
